In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
import pandas as pd
from sklearn.preprocessing import LabelEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE   = 224
BATCH_SIZE = 8
LR         = 1e-4
EPOCHS     = 50

TRAIN_CSV = os.environ.get("BIOMASS_TRAIN_CSV", "data/train.csv")
TEST_CSV  = os.environ.get("BIOMASS_TEST_CSV", "data/test.csv")
DATA_ROOT = os.environ.get("BIOMASS_DATA_ROOT", "data")
DINO_CKPT = os.environ.get("DINO_CHECKPOINT", "checkpoints/dino_deitsmall16_pretrain.pth")

LOSS_W = {
    "biomass": 1.0,
    "height":  0.2,
    "aux":     0.2,
    "species": 0.1,
    "state":   0.1,
}

TARGET_NAME_TO_ID = {
    "Dry_Clover_g": 0,
    "Dry_Dead_g": 1,
    "Dry_Green_g": 2,
    "Dry_Total_g": 3,
    "GDM_g": 4,
}
NUM_TARGET_NAMES = len(TARGET_NAME_TO_ID)

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

species_encoder = LabelEncoder()
state_encoder   = LabelEncoder()

train_df["Species"] = species_encoder.fit_transform(train_df["Species"])
train_df["State"]   = state_encoder.fit_transform(train_df["State"])

NUM_SPECIES = train_df["Species"].nunique()
NUM_STATES  = train_df["State"].nunique()

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

def patch_to_tiles(x, tile_h, tile_w):
    B, H, W, D = x.shape
    x = x.view(
        B,
        H // tile_h, tile_h,
        W // tile_w, tile_w,
        D
    )
    x = x.mean(dim=(2, 4))
    return x.view(B, -1, D)

class VegetationIndex(nn.Module):
    def forward(self, img):
        R, G, B = img[:, 0], img[:, 1], img[:, 2]
        exg = 2 * G - R - B
        exg = F.interpolate(
            exg.unsqueeze(1),
            size=(14, 14),
            mode="bilinear",
            align_corners=False
        )
        return exg.squeeze(1)

class CrossAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, q, kv):
        out, _ = self.attn(q, kv, kv)
        return self.norm(out + q)

class BiomassModel(nn.Module):
    def __init__(self, embed_dim=384, num_heads=6, target_emb_dim=128):
        super().__init__()

        self.dino = timm.create_model(
            "vit_small_patch16_224",
            pretrained=False,
            num_classes=0,
        )
        ckpt = torch.load(DINO_CKPT, map_location="cpu")
        self.dino.load_state_dict(ckpt, strict=False)

        for p in self.dino.parameters():
            p.requires_grad = False

        self.veg = VegetationIndex()

        self.local_proj = nn.Sequential(
            nn.Linear(embed_dim + 1, embed_dim),
            nn.ReLU(),
            nn.LayerNorm(embed_dim),
        )
        self.global_proj = nn.Sequential(
            nn.Linear(embed_dim + 1, embed_dim),
            nn.ReLU(),
            nn.LayerNorm(embed_dim),
        )

        self.local_attn  = CrossAttention(embed_dim, num_heads)
        self.global_attn = CrossAttention(embed_dim, num_heads)

        self.token_pool = nn.Linear(embed_dim, 1)

        self.target_embedding = nn.Embedding(NUM_TARGET_NAMES, target_emb_dim)
        self.target_proj = nn.Linear(target_emb_dim, embed_dim)

        fused_dim = embed_dim * 3 + embed_dim  # CLS + AVG + MAX + TARGET

        self.biomass_head = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )
        self.height_head  = nn.Linear(fused_dim, 1)
        self.aux_head     = nn.Linear(fused_dim, 1)
        self.species_head = nn.Linear(fused_dim, NUM_SPECIES)
        self.state_head   = nn.Linear(fused_dim, NUM_STATES)

    def forward(self, image, target_name_id):
        B = image.size(0)

        tokens = self.dino.forward_features(image)[:, 1:, :]
        tokens = tokens.view(B, 14, 14, -1)

        local_tokens  = patch_to_tiles(tokens, 2, 2)
        global_tokens = patch_to_tiles(tokens, 7, 7)

        exg = self.veg(image).unsqueeze(-1)
        exg_local  = patch_to_tiles(exg, 2, 2)
        exg_global = patch_to_tiles(exg, 7, 7)

        local_tokens  = self.local_proj(torch.cat([local_tokens, exg_local], dim=-1))
        global_tokens = self.global_proj(torch.cat([global_tokens, exg_global], dim=-1))

        global_tokens = self.global_attn(global_tokens, local_tokens)
        local_tokens  = self.local_attn(local_tokens, global_tokens)

        all_tokens = torch.cat([global_tokens, local_tokens], dim=1)

        w = F.softmax(self.token_pool(all_tokens), dim=1)
        cls_like = (w * all_tokens).sum(dim=1)
        avg_pool = all_tokens.mean(dim=1)
        max_pool = all_tokens.max(dim=1)[0]

        img_ctx = torch.cat([cls_like, avg_pool, max_pool], dim=1)

        tgt = self.target_proj(self.target_embedding(target_name_id))
        fused = torch.cat([img_ctx, tgt], dim=1)

        return {
            "biomass": self.biomass_head(fused).squeeze(1),
            "height":  self.height_head(fused).squeeze(1),
            "aux":     self.aux_head(fused).squeeze(1),
            "species": self.species_head(fused),
            "state":   self.state_head(fused),
        }

class BiomassDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = Image.open(os.path.join(DATA_ROOT, r["image_path"])).convert("RGB")
        img = self.transform(img)

        return {
            "image": img,
            "target": torch.tensor(r["target"], dtype=torch.float32),
            "target_name": torch.tensor(TARGET_NAME_TO_ID[r["target_name"]]),
            "species": torch.tensor(r["Species"]),
            "state": torch.tensor(r["State"]),
            "height": torch.tensor(r.get("Height_Ave_cm", 0.0), dtype=torch.float32),
            "aux": torch.tensor(r.get("Pre_GSHH_NDVI", 0.0), dtype=torch.float32),
        }

    def __len__(self):
        return len(self.df)

class BiomassTestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = Image.open(os.path.join(DATA_ROOT, r["image_path"])).convert("RGB")
        img = self.transform(img)

        return {
            "image": img,
            "target_name": torch.tensor(TARGET_NAME_TO_ID[r["target_name"]]),
            "id": r["sample_id"],
        }

    def __len__(self):
        return len(self.df)

def train_one_epoch(model, loader, optimizer):
    model.train()
    total = 0.0

    for b in loader:
        out = model(
            b["image"].to(DEVICE),
            b["target_name"].to(DEVICE)
        )

        loss = (
            LOSS_W["biomass"] * F.mse_loss(out["biomass"], b["target"].to(DEVICE)) +
            LOSS_W["height"]  * F.mse_loss(out["height"], b["height"].to(DEVICE)) +
            LOSS_W["aux"]     * F.mse_loss(out["aux"], b["aux"].to(DEVICE)) +
            LOSS_W["species"] * F.cross_entropy(out["species"], b["species"].to(DEVICE)) +
            LOSS_W["state"]   * F.cross_entropy(out["state"], b["state"].to(DEVICE))
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()

    return total / len(loader)

train_ds = BiomassDataset(train_df, transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

model = BiomassModel().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)



EPOCHS = 100;
for e in range(EPOCHS):
    loss = train_one_epoch(model, train_loader, optimizer)
    print(f"Epoch {e+1}/{EPOCHS} | Loss: {loss:.4f}")

    if (e + 1) % 10 == 0:
        ckpt_path = f"biomass_epoch_{e+1}.pth"
        torch.save(model.state_dict(), ckpt_path)
        print(f"💾 Saved checkpoint: {ckpt_path}")

torch.save(model.state_dict(), "biomass_final.pth")
print("✅ Training complete")
